# Nhóm 6 — Verifier-Guided Repair: thực nghiệm trên Kaggle

**Cách chạy:** *Save Version → Save & Run All* (chạy nền tối đa 12 giờ, đóng trình duyệt được).
**Settings:** Accelerator = *None* (CPU, 4 vCPU, ~30 GB RAM) · **Internet = On**.
**Secrets** (Add-ons → Secrets): `GEMINI_API_KEY`, tick cho notebook này.

Chỉ cần sửa **cell CẤU HÌNH** bên dưới giữa các lần chạy:
- **Lần 1:** để mặc định → cài đặt, sinh dữ liệu, thí nghiệm miễn phí, pilot LLM (~1.300đ).
- **Lần 2** (sau khi trưởng nhóm duyệt dự toán): `RUN_FREE = False`, `RUN_LLM_MAIN = True`.
  Nên *Add Input → Notebook Output* chọn version lần 1 để dùng lại dataset (khỏi sinh lại ~1 giờ).

Notebook không bao giờ dừng giữa chừng: bước lưu kết quả cuối cùng luôn chạy → tải `out.zip` ở tab **Output**.

## CẤU HÌNH

In [ ]:
RUN_DATA     = True    # sinh seed + dataset (tự bỏ qua nếu tìm thấy dataset từ version trước trong /kaggle/input)
RUN_FREE     = True    # thí nghiệm chỉ dùng Lean ($0): sàn, rule-only, scalability 100/1K/10K x 1/2/4 worker x 3 lần
RUN_PILOT    = True    # pilot LLM 50+50 mẫu (~1.300đ) + dự toán
RUN_LLM_MAIN = False   # thí nghiệm LLM chính — CHỈ bật sau khi trưởng nhóm duyệt dự toán
BRANCH = "feat/verifier-guided-pipeline"

## 0. Lấy code

In [ ]:
import os, glob, subprocess
os.chdir("/root")
!rm -rf repo && git clone -q --depth 1 -b {BRANCH} https://github.com/tamdv/ai4math-verifier-guided-repair.git repo
os.chdir("/root/repo")
!git log --oneline -1
os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]

## 1. Cài môi trường (~10–15 phút)

In [ ]:
!bash scripts/kaggle_setup.sh
!python scripts/check_llm.py   # 1 lần gọi rất nhỏ để kiểm tra key

## 2. Dữ liệu

In [ ]:
prev = sorted(glob.glob("/kaggle/input/*/out/errors/errors_10k.jsonl"))
if prev:
    src = os.path.dirname(os.path.dirname(prev[-1]))
    print("Dùng lại dữ liệu từ version trước:", src)
    !mkdir -p data/errors data/seeds && cp {src}/errors/*.jsonl data/errors/ && cp {src}/seeds_verified.jsonl {src}/natural_errors.jsonl data/seeds/ 2>/dev/null; true
elif RUN_DATA:
    !python scripts/build_seeds.py --n 3000 --workers 4
    !python scripts/check_strip_comments.py --workers 4
    !python scripts/generate_error_dataset.py --workers 4 --max-per-seed 5
!wc -l data/errors/*.jsonl

## 3. Thí nghiệm miễn phí (chỉ Lean)

In [ ]:
if RUN_FREE:
    !python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --strategy none --budget 0 --workers 4 --tag floor
    !python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --strategy rule_only --budget 3 --workers 4 --tag rule
    # Scalability: 3 mức dữ liệu (100 -> 1K -> 10K) x 3 mức worker x 3 lần lặp
    for data in ["errors_100", "errors_1k", "errors_10k"]:
        for w in [1, 2, 4]:
            for r in [1, 2, 3]:
                !python scripts/run_experiment.py --data data/errors/{data}.jsonl --split all --strategy rule_only --budget 1 --workers {w} --repeat-index {r} --tag scale

## 4. Pilot LLM (~1.300đ) + dự toán — gửi output cell này cho trưởng nhóm

In [ ]:
if RUN_PILOT:
    !python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --limit 50 --strategy blind --budget 1 --workers 4 --run-cap 0.10 --tag pilot
    !python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --limit 50 --strategy llm_only --budget 1 --workers 4 --run-cap 0.10 --tag pilot
    !python scripts/estimate_cost.py --data data/errors/errors_10k.jsonl

## 5. Thí nghiệm LLM chính (chỉ khi RUN_LLM_MAIN = True)

In [ ]:
if RUN_LLM_MAIN:
    for r in [1, 2, 3]:   # lần 2-3 dùng cache câu trả lời => ~0đ
        for strat, b in [("blind", 1), ("llm_only", 1), ("hybrid", 1), ("llm_only", 3), ("hybrid", 3)]:
            !python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --strategy {strat} --budget {b} --workers 4 --repeat-index {r} --tag rq
    !python scripts/run_experiment.py --data data/errors/errors_10k.jsonl --split holdout --strategy hybrid --budget 1 --workers 4 --run-cap 2.0 --tag scale-llm
    !python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --strategy llm_only --budget 1 --workers 4 --model gemini-3.6-flash --tag strong-model
else:
    print("RUN_LLM_MAIN = False -> bỏ qua (chưa duyệt dự toán)")

## 6. Lưu kết quả (luôn chạy)
Tải `out.zip` ở tab **Output** và gửi lại để commit.

In [ ]:
!mkdir -p /kaggle/working/out/errors && cp -r results /kaggle/working/out/ 2>/dev/null; cp data/errors/*.jsonl /kaggle/working/out/errors/ 2>/dev/null; cp data/seeds/*.jsonl /kaggle/working/out/ 2>/dev/null; cp data_sample/errors_sample_100.jsonl /kaggle/working/out/ 2>/dev/null; true
!cd /kaggle/working && zip -qr out.zip out && ls -lh out.zip
!ls results/raw | tail -40
!tail -3 results/cost_ledger.jsonl 2>/dev/null || echo "Không có lần gọi trả phí nào" 